In [ ]:
import pyarrow.parquet as pq

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/ReplaceBG.parquet'
pf = pq.ParquetFile(path)

total_rows = pf.metadata.num_rows
null_counts = {col: 0 for col in pf.schema_arrow.names}

# Read in manageable batches
for batch in pf.iter_batches(batch_size=100_000):
    for col in batch.schema.names:
        null_counts[col] += batch.column(col).null_count

print("=" * 80)
print(f"AZT1D — EMPTY VALUES PER COLUMN")
print(f"Total rows: {total_rows:,}")
print("=" * 80)

results = []

for col, null_count in null_counts.items():
    percentage = (null_count / total_rows) * 100

    results.append((col, null_count, percentage))

# Sort from most empty to least empty
results.sort(key=lambda x: x[2], reverse=True)

for col, null_count, percentage in results:
    print(f"{col:<40} {null_count:>12,} empty | {percentage:>8.2f}%")

AZT1D — EMPTY VALUES PER COLUMN
Total rows: 20,864,819
CGM                                         7,164,405 empty |    34.34%
carbs                                       2,641,828 empty |    12.66%
basal                                       1,458,378 empty |     6.99%
insulin                                     1,458,375 empty |     6.99%
bolus                                       1,332,854 empty |     6.39%
insulin_type_basal                          1,272,298 empty |     6.10%
insulin_type_bolus                          1,272,298 empty |     6.10%
ethnicity                                     572,429 empty |     2.74%
cgm_device                                    488,428 empty |     2.34%
insulin_delivery_algorithm                    488,428 empty |     2.34%
insulin_delivery_device                       488,428 empty |     2.34%
insulin_delivery_modality                     488,428 empty |     2.34%
is_pregnant                                   488,428 empty |     2.34%
age_of_di

In [1]:
import pyarrow.parquet as pq
from collections import Counter

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/ReplaceBG.parquet'
pf = pq.ParquetFile(path)

values = Counter()

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["insulin_type_bolus"]
):
    for value in batch.column("insulin_type_bolus").to_pylist():
        if value is not None:
            values[value] += 1

print("Number of different values:", len(values))

print("\nDifferent insulin_type_bolus values:")
for value, count in values.most_common():
    print(f"{value}: {count}")

Number of different values: 3

Different insulin_type_bolus values:
Novolog (Aspart): 11119517
Humalog (Lispro): 8213252
Apidra (Glulisine): 259752


In [1]:
import pyarrow.parquet as pq
import pandas as pd

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/ReplaceBG.parquet'

pf = pq.ParquetFile(path)

# Read a small sample
table = pf.read_row_group(0, columns=[
    "date",
    "CGM",
    "carbs",
    "basal",
    "insulin",
    "bolus",
    "id"
])

df = table.to_pandas()

# Show rows where bolus is NaN
print("Rows where bolus is NaN:")
print(df[df["bolus"].isna()].head(30).to_string(index=False))

print("\n" + "="*80)

# Show rows where bolus is explicitly 0
print("Rows where bolus = 0:")
print(df[df["bolus"] == 0].head(30).to_string(index=False))

Rows where bolus is NaN:
               date  CGM  carbs    basal  insulin  bolus id
2013-09-28 03:35:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 03:40:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 03:45:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 03:50:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 03:55:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 04:00:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 04:05:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 04:10:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 04:15:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 04:20:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 04:25:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 04:30:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 04:35:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 04:40:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 04:45:00  NaN    NaN 0.054167 0.054167    NaN 10
2013-09-28 04:5

In [ ]:
import pyarrow.parquet as pq
import pandas as pd

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/ReplaceBG.parquet'

pf = pq.ParquetFile(path)

nan_counts = {col: 0 for col in pf.schema.names}
total_rows = pf.metadata.num_rows

for batch in pf.iter_batches(batch_size=100_000):
    df = batch.to_pandas()

    for col in df.columns:
        nan_counts[col] += df[col].isna().sum()

result = pd.DataFrame({
    "Column": nan_counts.keys(),
    "NaN Count": nan_counts.values(),
})

result["NaN Percentage"] = (
    result["NaN Count"] / total_rows * 100
)

result = result.sort_values("NaN Percentage", ascending=False)

print(f"Total rows: {total_rows:,}")
print("=" * 80)
print(result.to_string(index=False))

Total rows: 20,864,819
                        Column  NaN Count  NaN Percentage
                           CGM    7164405       34.337250
                         carbs    2641828       12.661639
                         basal    1458378        6.989651
                       insulin    1458375        6.989636
                         bolus    1332854        6.388045
            insulin_type_bolus    1272298        6.097815
            insulin_type_basal    1272298        6.097815
                     ethnicity     572429        2.743513
       insulin_delivery_device     488428        2.340917
              age_of_diagnosis     488428        2.340917
                   is_pregnant     488428        2.340917
     insulin_delivery_modality     488428        2.340917
    insulin_delivery_algorithm     488428        2.340917
                    cgm_device     488428        2.340917
                           age          0        0.000000
                            id          0        

In [3]:
import pyarrow.parquet as pq
import pyarrow as pa

input_path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/ReplaceBG.parquet'

output_path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/ReplaceBGdropped.parquet'

# Columns to drop
drop_cols = [
    "is_pregnant",
    "ethnicity",
    "insulin_delivery_device",
    "cgm_device",
    "is_test",
    "source_file",
    "subject_split_across_traintest",
    "patient_stream_id",
]

pf = pq.ParquetFile(input_path)

# Get remaining columns
remaining_cols = [
    col for col in pf.schema.names
    if col not in drop_cols
]

print("Columns being kept:")
for col in remaining_cols:
    print(" -", col)

print("\nColumns being dropped:")
for col in drop_cols:
    print(" -", col)

# Create writer using the original schema minus dropped columns
first_table = pf.read_row_group(0, columns=remaining_cols)

writer = pq.ParquetWriter(
    output_path,
    first_table.schema,
    compression="snappy"
)

# Write first row group
writer.write_table(first_table)

# Write remaining row groups
for i in range(1, pf.num_row_groups):
    table = pf.read_row_group(i, columns=remaining_cols)
    writer.write_table(table)

    print(f"Processed row group {i + 1}/{pf.num_row_groups}")

writer.close()

print("\nDone!")
print(f"Saved to:\n{output_path}")

Columns being kept:
 - CGM
 - age
 - basal
 - bolus
 - carbs
 - date
 - height
 - id
 - insulin
 - insulin_delivery_algorithm
 - insulin_delivery_modality
 - insulin_type_basal
 - insulin_type_bolus
 - weight
 - gender
 - age_of_diagnosis

Columns being dropped:
 - is_pregnant
 - ethnicity
 - insulin_delivery_device
 - cgm_device
 - is_test
 - source_file
 - subject_split_across_traintest
 - patient_stream_id
Processed row group 2/209
Processed row group 3/209
Processed row group 4/209
Processed row group 5/209
Processed row group 6/209
Processed row group 7/209
Processed row group 8/209
Processed row group 9/209
Processed row group 10/209
Processed row group 11/209
Processed row group 12/209
Processed row group 13/209
Processed row group 14/209
Processed row group 15/209
Processed row group 16/209
Processed row group 17/209
Processed row group 18/209
Processed row group 19/209
Processed row group 20/209
Processed row group 21/209
Processed row group 22/209
Processed row group 23/209
P

In [4]:
import pyarrow.parquet as pq
import pandas as pd

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/ReplaceBGdropped.parquet'

pf = pq.ParquetFile(path)
total_rows = pf.metadata.num_rows

counts = {
    "CGM available": 0,
    "Carbs available": 0,
    "Bolus available": 0,
    "CGM + Carbs available": 0,
    "CGM + Bolus available": 0,
    "Carbs + Bolus available": 0,
    "CGM + Carbs + Bolus available": 0,
}

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["CGM", "carbs", "bolus"]
):
    df = batch.to_pandas()

    cgm = df["CGM"].notna()
    carbs = df["carbs"].notna()
    bolus = df["bolus"].notna()

    counts["CGM available"] += cgm.sum()
    counts["Carbs available"] += carbs.sum()
    counts["Bolus available"] += bolus.sum()

    counts["CGM + Carbs available"] += (cgm & carbs).sum()
    counts["CGM + Bolus available"] += (cgm & bolus).sum()
    counts["Carbs + Bolus available"] += (carbs & bolus).sum()
    counts["CGM + Carbs + Bolus available"] += (cgm & carbs & bolus).sum()

print(f"Total rows: {total_rows:,}")
print("=" * 80)

for name, count in counts.items():
    percentage = count / total_rows * 100
    print(f"{name:<35} {count:>15,}  ({percentage:6.2f}%)")

Total rows: 20,864,819
CGM available                            13,700,414  ( 65.66%)
Carbs available                          18,222,991  ( 87.34%)
Bolus available                          19,531,965  ( 93.61%)
CGM + Carbs available                    12,512,900  ( 59.97%)
CGM + Bolus available                    13,256,924  ( 63.54%)
Carbs + Bolus available                  18,220,423  ( 87.33%)
CGM + Carbs + Bolus available            12,510,383  ( 59.96%)


In [5]:
import pyarrow.parquet as pq
import pandas as pd

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/ReplaceBGdropped.parquet'

pf = pq.ParquetFile(path)
total_rows = pf.metadata.num_rows

counts = {
    "carbs_nan": 0,
    "carbs_zero": 0,
    "carbs_positive": 0,
    "bolus_nan": 0,
    "bolus_zero": 0,
    "bolus_positive": 0,
    
    "bolus_zero_insulin_equals_basal": 0,
    "bolus_zero_with_values": 0,
    
    "bolus_positive_insulin_matches_sum": 0,
    "bolus_positive_with_values": 0,
}

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["carbs", "bolus", "basal", "insulin"]
):
    df = batch.to_pandas()

    # -------------------------
    # CARBS
    # -------------------------
    carbs_nan = df["carbs"].isna()
    carbs_zero = df["carbs"].notna() & (df["carbs"] == 0)
    carbs_positive = df["carbs"] > 0

    counts["carbs_nan"] += carbs_nan.sum()
    counts["carbs_zero"] += carbs_zero.sum()
    counts["carbs_positive"] += carbs_positive.sum()

    # -------------------------
    # BOLUS
    # -------------------------
    bolus_nan = df["bolus"].isna()
    bolus_zero = df["bolus"].notna() & (df["bolus"] == 0)
    bolus_positive = df["bolus"] > 0

    counts["bolus_nan"] += bolus_nan.sum()
    counts["bolus_zero"] += bolus_zero.sum()
    counts["bolus_positive"] += bolus_positive.sum()

    # -------------------------
    # BOLUS = 0
    # Check insulin ≈ basal
    # -------------------------
    valid_zero = (
        bolus_zero &
        df["insulin"].notna() &
        df["basal"].notna()
    )

    counts["bolus_zero_with_values"] += valid_zero.sum()

    counts["bolus_zero_insulin_equals_basal"] += (
        valid_zero &
        ((df["insulin"] - df["basal"]).abs() < 1e-6)
    ).sum()

    # -------------------------
    # BOLUS > 0
    # Check insulin ≈ basal + bolus
    # -------------------------
    valid_positive = (
        bolus_positive &
        df["insulin"].notna() &
        df["basal"].notna()
    )

    counts["bolus_positive_with_values"] += valid_positive.sum()

    counts["bolus_positive_insulin_matches_sum"] += (
        valid_positive &
        ((df["insulin"] - (df["basal"] + df["bolus"])).abs() < 1e-6)
    ).sum()


# ============================================================
# PRINT RESULTS
# ============================================================

print(f"Total rows: {total_rows:,}")
print("=" * 90)

print("\nCARBS")
print("-" * 90)

for key in ["carbs_nan", "carbs_zero", "carbs_positive"]:
    count = counts[key]
    print(f"{key:<25} {count:>15,}  ({count / total_rows * 100:6.2f}%)")


print("\nBOLUS")
print("-" * 90)

for key in ["bolus_nan", "bolus_zero", "bolus_positive"]:
    count = counts[key]
    print(f"{key:<25} {count:>15,}  ({count / total_rows * 100:6.2f}%)")


print("\nBOLUS = 0 → INSULIN vs BASAL")
print("-" * 90)

print(
    f"Rows with bolus=0 and insulin/basal available: "
    f"{counts['bolus_zero_with_values']:,}"
)

print(
    f"Rows where insulin ≈ basal: "
    f"{counts['bolus_zero_insulin_equals_basal']:,}"
)

if counts["bolus_zero_with_values"] > 0:
    pct = (
        counts["bolus_zero_insulin_equals_basal"]
        / counts["bolus_zero_with_values"]
        * 100
    )
    print(f"Percentage: {pct:.2f}%")


print("\nBOLUS > 0 → INSULIN vs BASAL + BOLUS")
print("-" * 90)

print(
    f"Rows with bolus>0 and insulin/basal available: "
    f"{counts['bolus_positive_with_values']:,}"
)

print(
    f"Rows where insulin ≈ basal + bolus: "
    f"{counts['bolus_positive_insulin_matches_sum']:,}"
)

if counts["bolus_positive_with_values"] > 0:
    pct = (
        counts["bolus_positive_insulin_matches_sum"]
        / counts["bolus_positive_with_values"]
        * 100
    )
    print(f"Percentage: {pct:.2f}%")

Total rows: 20,864,819

CARBS
------------------------------------------------------------------------------------------
carbs_nan                       2,641,828  ( 12.66%)
carbs_zero                     18,013,281  ( 86.33%)
carbs_positive                    209,710  (  1.01%)

BOLUS
------------------------------------------------------------------------------------------
bolus_nan                       1,332,854  (  6.39%)
bolus_zero                     18,884,867  ( 90.51%)
bolus_positive                    647,098  (  3.10%)

BOLUS = 0 → INSULIN vs BASAL
------------------------------------------------------------------------------------------
Rows with bolus=0 and insulin/basal available: 18,575,306
Rows where insulin ≈ basal: 18,575,306
Percentage: 100.00%

BOLUS > 0 → INSULIN vs BASAL + BOLUS
------------------------------------------------------------------------------------------
Rows with bolus>0 and insulin/basal available: 639,756
Rows where insulin ≈ basal + bolus: 639,7